# 06 — Classical Unsupervised ML


## Objective
Fit Isolation Forest, Local Outlier Factor and Elliptic Envelope on the training feature matrix and evaluate ranking quality on the validation set (labels used only for metrics).


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import joblib

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_processed
from anomaly import fit_isolation_forest, fit_lof, fit_elliptic_envelope
from evaluation import evaluate_anomaly_scores, summary_table

X_train = load_processed("X_train").drop(columns=["class"])
X_val   = load_processed("X_val")
y_val   = X_val["class"].values
X_val_f = X_val.drop(columns=["class"])

print("Fitting IsolationForest …")
iforest = fit_isolation_forest(X_train, contamination=0.09, n_estimators=200)
print("Fitting LOF …")
lof = fit_lof(X_train, n_neighbors=25, contamination=0.09)
print("Fitting EllipticEnvelope …")
ee = fit_elliptic_envelope(X_train, contamination=0.09)

results = {}
for name, model in [("IsolationForest", iforest), ("LOF", lof), ("EllipticEnvelope", ee)]:
    s = model.predict_anomaly_score(X_val_f)
    results[name] = evaluate_anomaly_scores(y_val, s)

print(summary_table(results))

# Persist the best model (IsolationForest is usually strongest)
(Path(ROOT) / "models").mkdir(exist_ok=True)
joblib.dump(iforest, ROOT / "models" / "isolation_forest.joblib")
print("Saved IsolationForest to models/")
